# 03 - Portfolio Construction

Compare **HRP** vs **mean-variance** vs **minimum-variance** vs **risk-parity** vs **equal-weight** on an S&P-100-style basket, then backtest each with mandatory transaction costs and rank by Sharpe.

> **Data input:** Set exactly one of `QUANTCORTEX_PRICES_CSV` or `QUANTCORTEX_LIVE_YFINANCE=1`. No market data or executed outputs are committed.


In [ ]:
import logging
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

logging.getLogger("hmmlearn").setLevel(logging.ERROR)
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import matplotlib

try:
    shell = get_ipython()
except NameError:
    shell = None
if shell is None:
    matplotlib.use("Agg")
else:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

# Put the repository root on the path when running from research/.
for candidate in ("..", "."):
    absolute = os.path.abspath(candidate)
    if absolute not in sys.path:
        sys.path.insert(0, absolute)

from quantcortex.backtest.metrics.plotting import (
    COUNTERFACTUAL_AMBER, INK, NEGATIVE_RED, REFERENCE_BLUE,
    apply_plot_style, style_axis,
)
from quantcortex.data.local_csv import load_ohlcv_csv, load_price_matrix

apply_plot_style("notebook")

PRICE_CSV = os.environ.get("QUANTCORTEX_PRICES_CSV")
OHLCV_CSV = os.environ.get("QUANTCORTEX_OHLCV_CSV")
LIVE_YFINANCE = os.environ.get("QUANTCORTEX_LIVE_YFINANCE") == "1"

if (PRICE_CSV is not None) == LIVE_YFINANCE:
    raise RuntimeError(
        "Set exactly one of QUANTCORTEX_PRICES_CSV=/path/to/prices.csv "
        "or QUANTCORTEX_LIVE_YFINANCE=1."
    )


def load_prices(symbols, start="2018-01-01"):
    """Load adjusted closes from the explicitly selected real-data source."""
    if PRICE_CSV is not None:
        return load_price_matrix(PRICE_CSV, symbols=list(symbols), start=start)

    print(
        "Using live Yahoo Finance data through yfinance; review the provider "
        "terms and confirm that your use is permitted."
    )
    from quantcortex.data.providers.yfinance_provider import YFinanceProvider

    prices = YFinanceProvider().get_prices(list(symbols), start=start)
    if prices is None or prices.empty:
        raise RuntimeError("yfinance returned no prices")
    prices = prices.dropna(how="all").ffill(limit=5).dropna()
    if prices.shape[0] <= 120:
        raise RuntimeError("insufficient complete price history from yfinance")
    return prices


def load_ohlcv(symbol, start="2018-01-01"):
    """Load one symbol's real OHLCV data for feature engineering."""
    if OHLCV_CSV is not None:
        return load_ohlcv_csv(OHLCV_CSV, start=start)
    if not LIVE_YFINANCE:
        raise RuntimeError(
            "Set QUANTCORTEX_OHLCV_CSV=/path/to/ohlcv.csv for the Alpha158 cell."
        )

    from quantcortex.data.providers.yfinance_provider import YFinanceProvider

    frame = YFinanceProvider().fetch_ohlcv([symbol], start=start).get(symbol)
    if frame is None or frame.empty:
        raise RuntimeError(f"yfinance returned no OHLCV data for {symbol}")
    return frame.dropna()


selected_source = (
    f"local CSV {Path(PRICE_CSV).expanduser().resolve()}"
    if PRICE_CSV is not None
    else "explicit live yfinance download"
)
print(f"quantcortex research environment ready; source: {selected_source}")


In [ ]:
symbols = ["AAPL","MSFT","AMZN","NVDA","GOOGL","JPM","XOM","JNJ","PG","KO","UNH","HD"]
prices = load_prices(symbols)
returns = prices.pct_change(fill_method=None).dropna()
print("basket:", len(symbols), "names |", prices.shape[0], "days")

## Fit the optimizers

In [ ]:
from quantcortex.portfolio.equal_weight import EqualWeight
from quantcortex.portfolio.hrp import HierarchicalRiskParity
from quantcortex.portfolio.mean_variance import MeanVariance
from quantcortex.portfolio.minimum_variance import MinimumVariance
from quantcortex.portfolio.risk_parity import RiskParity

optimizers = {
    "EqualWeight": EqualWeight(),
    "MeanVariance": MeanVariance(),
    "MinVariance": MinimumVariance(),
    "RiskParity": RiskParity(),
    "HRP": HierarchicalRiskParity(),
}
weights = {name: opt.optimize(returns) for name, opt in optimizers.items()}
wdf = pd.DataFrame(weights, index=symbols)
for name, w in weights.items():
    assert abs(w.sum() - 1.0) < 1e-6 and (w >= -1e-9).all()
print("all optimizers honor the weight contract (sum=1, long-only)")
wdf.round(3)

## Backtest each allocation (costs mandatory)

In [ ]:
from quantcortex.backtest.costs.transaction_costs import TransactionCostModel
from quantcortex.backtest.engines.vectorized import VectorizedBacktest

rebal = prices.resample("MS").first().index            # monthly rebalance
rebal = rebal[(rebal >= prices.index[0]) & (rebal <= prices.index[-1])]
cost_model = TransactionCostModel()
results, equity = {}, {}
for name, w in weights.items():
    panel = pd.DataFrame([w] * len(rebal), index=rebal, columns=symbols)
    res = VectorizedBacktest(cost_model).run(panel, prices)
    results[name] = res.summary(); equity[name] = res.equity_curve
summary = pd.DataFrame(results).T[["cagr","ann_vol","sharpe","max_drawdown"]]
summary.sort_values("sharpe", ascending=False).round(3)

In [ ]:
from matplotlib.ticker import FuncFormatter

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
pd.DataFrame(equity).plot(ax=ax[0], lw=1.3)
line_styles = ("-", "--", "-.", ":", (0, (5, 2)))
for line, line_style in zip(ax[0].lines, line_styles):
    line.set_linestyle(line_style)
ax[0].set_title("Portfolio value paths")
ax[0].set_xlabel("Date")
ax[0].set_ylabel("Portfolio value")
ax[0].yaxis.set_major_formatter(
    FuncFormatter(lambda value, _: f"${value / 1_000_000:.2f}M")
)
style_axis(ax[0], grid="y")
wdf[["HRP", "MeanVariance"]].plot(
    kind="barh", ax=ax[1], color=[REFERENCE_BLUE, COUNTERFACTUAL_AMBER]
)
for container, hatch in zip(ax[1].containers, ("", "///")):
    for bar in container:
        bar.set_edgecolor(INK if hatch else "white")
        bar.set_linewidth(0.4)
        bar.set_hatch(hatch)
ax[1].set_title("HRP versus mean-variance weights")
ax[1].set_xlabel("Portfolio weight")
ax[1].set_ylabel("Asset")
style_axis(ax[1], grid="x")
plt.tight_layout()
plt.show()
print("portfolio construction complete.")